# Stage 04c: Feature Encoding + Model Preparation

**Purpose:** Prepare ALL model-ready data

**Inputs:** data/04b_train.parquet, data/04b_test.parquet

**Outputs:**
- data/04c_train_encoded.parquet (after exclusions)
- data/04c_test_encoded.parquet (after exclusions)
- data/04c_train_base_margin.parquet (GLM predictions)
- data/04c_test_base_margin.parquet (GLM predictions)
- config_generated/04c_monotonicity_constraints.yaml

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import pandas as pd
import yaml
import os, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "lib"))
from utils import setup_notebook_environment, get_machine_config
from feature_encoder import apply_master_encoding, save_encoders
from model_utils import load_monotonicity_constraints, load_exclusions, load_glm_init, apply_feature_filters

print("########################################")
print("# STAGE 04c: ENCODING + MODEL PREP")
print("########################################")

project_root = setup_notebook_environment()

In [ ]:
config_file = f'{config_path}/config.yaml'
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

# Get current machine ID and output path
pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'  # current.pc has '3', config key is 'PC3'
output_base = cfg['machines'][pc_id]['paths']['output_path']
print(f'Output: {output_base}')
vehicle_type = cfg["experiment"]["vehicle_type"]
target = cfg["experiment"]["target"]
join_key = cfg["data"]["join_key"]
print(f"Vehicle type: {vehicle_type}")

In [ ]:
# Load train/test splits
train_file = f'{output_base}/data/04b_train.parquet'
test_file = f'{output_base}/data/04b_test.parquet'

print(f'\n* Loading data...')
train = pd.read_parquet(train_file)
test = pd.read_parquet(test_file)

print(f'  Train: {train.shape}')
print(f'  Test: {test.shape}')

In [ ]:
# Apply master encoding
print(f'\n* Applying master feature encoding...')

train_encoded, test_encoded, encoders, encoding_summary = apply_master_encoding(
    train, test, config_path,
    min_frequency=0.01,  # 1% threshold
    min_count=50         # OR 50 observations
)

In [ ]:
# Display encoding summary
print(f'\n* Encoding Summary:')
print(f'\nEncoding types used:')
print(encoding_summary['encoding_type'].value_counts())

print(f'\nCategories with __OTHER__:')
print(encoding_summary[encoding_summary['has_other']== True][['original_column', 'n_categories']].head(10))

print(f'\nCategories with __MISSING__:')
print(encoding_summary[encoding_summary['has_missing'] == True][['original_column', 'n_categories']].head(10))

In [ ]:
# Apply feature exclusions
print(f"\n* Filtering features...")
exclusions = load_exclusions(config_path, vehicle_type)
all_features = list(train_encoded.columns)
filtered_features = apply_feature_filters(all_features, exclusions)
train_encoded = train_encoded[filtered_features]
test_encoded = test_encoded[filtered_features]
print(f"  {len(all_features)} -> {len(filtered_features)} features")

In [ ]:
# Prepare monotonicity constraints
print(f"\n* Preparing monotonicity constraints...")
mono_dict = load_monotonicity_constraints(config_path, vehicle_type, filtered_features)
os.makedirs(f"{output_base}/config_generated", exist_ok=True)
mono_file = f"{output_base}/config_generated/04c_monotonicity_constraints.yaml"
with open(mono_file, "w") as f:
    yaml.dump(mono_dict, f)
print(f"  Saved: {mono_file}")

In [ ]:
# Load GLM predictions as base_margin
if cfg.get("model", {}).get("use_glm_init", False):
    print(f"\n* Loading GLM predictions...")
    # Get machine-specific aux data path
    pc_num = open("current.pc").read().strip()
    pc_id = f"PC{pc_num}"
    aux_data_path = cfg["machines"][pc_id]["paths"]["aux_data_path"]
    control_file = cfg["data"]["control_model_file"]
    target_col = f"pred_{target}"
    transform = cfg["model"].get("glm_init_transform", "log")
    
    base_margin_train = load_glm_init(aux_data_path, control_file, train, join_key, target_col, transform)
    base_margin_test = load_glm_init(aux_data_path, control_file, test, join_key, target_col, transform)
    
    pd.DataFrame({"base_margin": base_margin_train}).to_parquet(f"{output_base}/data/04c_train_base_margin.parquet", index=False)
    pd.DataFrame({"base_margin": base_margin_test}).to_parquet(f"{output_base}/data/04c_test_base_margin.parquet", index=False)
    print(f"  Saved base_margin files")
else:
    print(f"\n* GLM init disabled")

In [ ]:
# Save encoded data
train_output = f'{output_base}/data/04c_train_encoded.parquet'
test_output = f'{output_base}/data/04c_test_encoded.parquet'

train_encoded.to_parquet(train_output, index=False)
test_encoded.to_parquet(test_output, index=False)

print(f'\n* Saved:')
print(f'  {train_output}')
print(f'  {test_output}')

In [ ]:
# Save encoders for holdout
save_encoders(encoders, encoding_summary, output_base)

In [ ]:
print("\n########################################")
print("# STAGE 04c: COMPLETE")
print("########################################")